# Dzień 1 — 10 krótkich ćwiczeń z AI

**Kolejność:** model bazowy → neuron → mała sieć → ocena → praca na plikach → Dask/DuckDB → self-attention → maska → wiele głów → Grid Search.

 Ćwiczenie 6 ma dwa opcjonalne dodatki (Dask i DuckDB), reszta działa z `numpy`, `pandas` i `scikit-learn`. Python 3.11+ i Jupyter wystarczą. Przed zajęciami można wykonać w terminalu:

```bash
python -m pip install numpy pandas scikit-learn jupyter
python -m pip install "dask[dataframe]" duckdb  # dodatki do ćwiczenia 6
```

Kod zapisuje trzy małe pliki CSV w folderze `ai_dzien1_dane` w bieżącym katalogu pracy Jupytera. Uruchamiaj komórki po kolei. Wyniki szybkości zależą od komputera; małe dane **nie służą do dowodzenia**, że Dask jest szybszy od Pandas.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

np.set_printoptions(precision=3, suppress=True)


## 1. Model bazowy przed siecią neuronową (8 min)

**Cel:** ustalić punkt odniesienia na danych nieliniowych. Tworzymy dwa przeplatające się półksiężyce, dzielimy dane z zachowaniem udziału klas i uczymy regresję logistyczną w pipeline ze skalowaniem.

**Sprawdź:** jaka jest skuteczność na zbiorze odłożonym? Dlaczego wynik na treningu nie wystarcza? Zmień `noise` z `0.22` na `0.35` i ponów próbę. Ten sam podział danych wykorzystamy w ćwiczeniach 4 i 10.


In [ ]:
X, y = make_moons(n_samples=500, noise=0.22, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
baseline.fit(X_train, y_train)
print('Baseline: accuracy train =', round(baseline.score(X_train, y_train), 3))
print('Baseline: accuracy test  =', round(baseline.score(X_test, y_test), 3))


## 2. Jeden neuron w NumPy (5 min)

**Cel:** zobaczyć iloczyn skalarny, bias i aktywację ReLU bez ukrywania obliczeń w bibliotece. Cztery wejścia odpowiadają kombinacjom dwóch bitów.

**Sprawdź:** które wejścia aktywują neuron? Zmień `b` na `-1.5`. Jedna liniowa granica decyzyjna nie rozwiąże XOR dla wszystkich czterech punktów.


In [ ]:
X_xor = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor = np.array([0, 1, 1, 0])
w = np.array([1., 1.])
b = -0.5
z = X_xor @ w + b
a = np.maximum(0, z)  # ReLU
print(pd.DataFrame({'x1': X_xor[:, 0], 'x2': X_xor[:, 1],
                    'z': z, 'ReLU(z)': a, 'etykieta_XOR': y_xor}))


## 3. Mała sieć uczy się XOR (8 min)

**Cel:** pokazać rolę warstwy ukrytej i nieliniowości. Model ma osiem neuronów ukrytych, a `lbfgs` dobrze nadaje się do tak małej próbki.

**Sprawdź:** porównaj cztery predykcje z etykietami. Zmień `hidden_layer_sizes` na `(1,)`. Czy wynik i stabilność między różnymi `random_state` są takie same? Przykład z czterema obserwacjami służy wyłącznie do pokazania mechanizmu, nie do oceny generalizacji.


In [ ]:
xor_net = MLPClassifier(hidden_layer_sizes=(8,), activation='tanh',
                        solver='lbfgs', max_iter=1000, random_state=4)
xor_net.fit(X_xor, y_xor)
print('Prawda:    ', y_xor)
print('Predykcja: ', xor_net.predict(X_xor))
print('Parametrów:', sum(W.size for W in xor_net.coefs_) +
                      sum(v.size for v in xor_net.intercepts_))


## 4. Sieć kontra model liniowy: ocena na teście (10 min)

**Cel:** porównać wyniki na identycznym podziale danych. Liczymy `accuracy`, `F1 macro` i macierz pomyłek, żeby sama skuteczność nie zasłoniła rodzaju błędów.

**Sprawdź:** które komórki macierzy pomyłek odpowiadają błędnie rozpoznanej klasie 1? Zmień szum w ćwiczeniu 1 i uruchom ponownie oba modele.


In [ ]:
network = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(16, 8), solver='lbfgs',
                  max_iter=1000, random_state=4)
)
network.fit(X_train, y_train)
for name, model in [('Regresja logistyczna', baseline), ('MLP', network)]:
    pred = model.predict(X_test)
    print(name, '| accuracy =', round(accuracy_score(y_test, pred), 3),
          '| F1 macro =', round(f1_score(y_test, pred, average='macro'), 3))
    print(confusion_matrix(y_test, pred), '\n')


## 5. Pandas: pliki czytane porcjami (10 min)

**Cel:** zrozumieć zasadę agregacji większej liczby plików bez ładowania ich naraz do jednego DataFrame. Wytwarzamy trzy lokalne CSV po 30 000 wierszy. To dane małe celowo, aby ćwiczenie trwało minuty.

**Sprawdź:** dlaczego średniej dla całej grupy nie wylicza się przez prostą średnią ze średnich porcjami? Poprawna agregacja przenosi sumę i liczebność.


In [ ]:
data_dir = Path('ai_dzien1_dane')
data_dir.mkdir(exist_ok=True)
rng = np.random.default_rng(21)
for i in range(3):
    frame = pd.DataFrame({
        'region': rng.choice(['A', 'B', 'C', 'D'], size=30_000),
        'value': rng.integers(1, 101, size=30_000),
    })
    frame.to_csv(data_dir / f'events_{i}.csv', index=False)

totals = {}
for file in sorted(data_dir.glob('events_*.csv')):
    for chunk in pd.read_csv(file, chunksize=10_000):
        stats = chunk.groupby('region')['value'].agg(['sum', 'count'])
        for region, row in stats.iterrows():
            sums, count = totals.get(region, (0, 0))
            totals[region] = (sums + int(row['sum']), count + int(row['count']))

pandas_means = {r: sums / count for r, (sums, count) in sorted(totals.items())}
print('Wierszy łącznie:', sum(count for _, count in totals.values()))
print('Średnie według regionów:', pandas_means)


## 6. Dask i DuckDB na tych samych plikach (10–12 min)

**Cel:** porównać trzy style pracy z tym samym zbiorem: Pandas porcjami, Dask z leniwym grafem zadań oraz SQL w DuckDB. Obie biblioteki są opcjonalne. Jeśli brak modułu, komórka pokaże polecenie instalacji i przejdzie dalej.

**Sprawdź:** czy średnie zgadzają się z ćwiczeniem 5? Co robi `.compute()` w Dask? Na małej próbce nie wyciągaj wniosków o przyspieszeniu, bo narzut harmonogramowania może dominować. Wariant dla prowadzącego: zwiększ liczbę plików dopiero po sprawdzeniu czasu na sali.


In [ ]:
from importlib.util import find_spec
csv_pattern = str(data_dir / 'events_*.csv')

if find_spec('dask'):
    import dask
    import dask.dataframe as dd
    # Użycie zwykłych kolumn tekstowych pozwala uruchomić przykład bez PyArrow.
    with dask.config.set({'dataframe.convert-string': False}):
        ddf = dd.read_csv(csv_pattern)
        lazy_stats = ddf.groupby('region')['value'].agg(['sum', 'count'])
        print('Dask: przed compute =', type(lazy_stats).__name__)
        dask_stats = lazy_stats.compute()
    print('Dask: średnie =', (dask_stats['sum'] / dask_stats['count']).to_dict())
else:
    print('Dask pominięty. Instalacja: python -m pip install "dask[dataframe]"')

if find_spec('duckdb'):
    import duckdb
    sql = """SELECT region, AVG(value) AS mean_value, COUNT(*) AS rows
             FROM read_csv('ai_dzien1_dane/events_*.csv')
             GROUP BY region ORDER BY region"""
    print('DuckDB:')
    print(duckdb.sql(sql).df())
else:
    print('DuckDB pominięty. Instalacja: python -m pip install duckdb')


## 7. Macierz self-attention w NumPy (10 min)

**Cel:** policzyć `Q`, `K`, `V`, macierz podobieństw i znormalizowane wagi dla trzech umownych tokenów. To ilustracja jednej głowy uwagi, bez wytrenowanych parametrów. Wiersz wag mówi, z których pozycji dany token korzysta w tej głowie.

**Sprawdź:** wymiary `Q @ K.T`, sumy wierszy i wynikową reprezentację. Zmień jeden embedding i zaobserwuj zmianę wag.


In [ ]:
tokens = ['czujnik', 'zgłasza', 'alarm']
E = np.array([[1., 0., 0., 1.],
              [0., 1., 1., 0.],
              [1., 1., 0., 0.]])  # 3 tokeny × 4 cechy
Wq = np.array([[1., 0.], [0., 1.], [0., 0.], [0., 0.]])
Wk = np.array([[1., 0.], [0., 1.], [0., 0.], [0., 0.]])
Wv = np.array([[1., 0.], [0., 1.], [0., 1.], [0., 0.]])
Q, K, V = E @ Wq, E @ Wk, E @ Wv
scores = Q @ K.T / np.sqrt(Q.shape[1])
def softmax_rows(a):
    shifted = a - np.max(a, axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)
weights = softmax_rows(scores)
context = weights @ V
print('QKᵀ:', scores.shape, '\n', pd.DataFrame(scores, index=tokens, columns=tokens))
print('Wagi:', '\n', pd.DataFrame(weights, index=tokens, columns=tokens).round(3))
print('Suma każdego wiersza:', weights.sum(axis=1))
print('Wynik:', context.shape, '\n', context)


## 8. Maska przyczynowa (8 min)

**Cel:** wymusić, by pozycja nie widziała tokenów z przyszłości. Wagi niedozwolonych połączeń powinny wynosić zero. Maskę stosujemy **przed** softmax.

**Sprawdź:** pierwszy wiersz powinien mieć wagę tylko na pierwszym tokenie. Usuń maskę i porównaj ostatnie dwie pozycje.


In [ ]:
future = np.triu(np.ones_like(scores, dtype=bool), k=1)
causal_scores = np.where(future, -np.inf, scores)
causal_weights = softmax_rows(causal_scores)
print(pd.DataFrame(causal_weights, index=tokens, columns=tokens).round(3))
print('Suma wierszy:', causal_weights.sum(axis=1))
assert np.allclose(causal_weights[future], 0)


## 9. Dwie głowy uwagi (8 min)

**Cel:** zobaczyć, że odrębne projekcje `Q`, `K`, `V` mogą dać różne macierze uwagi dla tych samych tokenów. Wyniki głów łączymy wzdłuż wymiaru cech. To minimalna demonstracja mechanizmu, **nie pełny model Transformer**.

**Sprawdź:** porównaj macierze wag obu głów. Co się stanie, gdy druga głowa otrzyma dokładnie te same projekcje co pierwsza?


In [ ]:
def attention_head(E, Wq, Wk, Wv):
    q, k, v = E @ Wq, E @ Wk, E @ Wv
    a = softmax_rows(q @ k.T / np.sqrt(q.shape[1]))
    return a, a @ v

a1, h1 = attention_head(E, Wq, Wk, Wv)
Wq2 = np.array([[0., 0.], [0., 0.], [1., 0.], [0., 1.]])
Wk2 = np.array([[0., 0.], [1., 0.], [0., 1.], [0., 0.]])
Wv2 = np.array([[0., 1.], [0., 0.], [1., 0.], [0., 0.]])
a2, h2 = attention_head(E, Wq2, Wk2, Wv2)
print('Głowa 1:', '\n', pd.DataFrame(a1, index=tokens, columns=tokens).round(2))
print('Głowa 2:', '\n', pd.DataFrame(a2, index=tokens, columns=tokens).round(2))
joined = np.concatenate([h1, h2], axis=1)
print('Połączenie głów:', joined.shape)


## 10. Grid Search dla MLP bez przecieku danych (12–15 min)

**Cel:** porównać cztery konfiguracje małej sieci w trzech foldach. Skalowanie umieszczamy w `Pipeline`, więc każde dopasowanie skaluje dane wyłącznie na części uczącej swojego foldu. Test odkładamy do końcowej oceny.

**Sprawdź:** `cv_results_` oraz `best_params_`. Ile modeli dopasowano? Dlaczego `best_score_` i wynik na `X_test` mogą się różnić? Zwiększ liczbę opcji dopiero po policzeniu kosztu.


In [ ]:
pipe = make_pipeline(
    StandardScaler(),
    MLPClassifier(solver='lbfgs', max_iter=500, random_state=4)
)
grid = {
    'mlpclassifier__hidden_layer_sizes': [(8,), (16, 8)],
    'mlpclassifier__alpha': [0.0001, 0.01],
}
search = GridSearchCV(pipe, grid, cv=3, scoring='f1_macro', n_jobs=1,
                      return_train_score=True)
search.fit(X_train, y_train)
print('Liczba konfiguracji:', len(search.cv_results_['params']))
print('Najlepsze parametry:', search.best_params_)
print('Średni F1 w CV:', round(search.best_score_, 3))
print('F1 na odłożonym teście:',
      round(f1_score(y_test, search.predict(X_test), average='macro'), 3))
print(pd.DataFrame(search.cv_results_)[
    ['param_mlpclassifier__hidden_layer_sizes', 'param_mlpclassifier__alpha',
     'mean_test_score', 'mean_train_score']].round(3))


## Wskazówki dla prowadzącego

- **Najkrótsza ścieżka na 60 minut:** 1, 2, 3, 4, 5, 7, 8, 10. Dask i DuckDB pokaż jako demonstrację, jeśli instalacja jest przygotowana.
- **Najważniejsze rozróżnienie:** ćwiczenia 7–9 używają sztucznych projekcji. Pokazują rachunek attention, ale nie znaczenie odpowiedzi wytrenowanego modelu.
- **Ostrzeżenie metodyczne:** cztery punkty XOR nie są sensownym zbiorem testowym. Do oceny generalizacji służą ćwiczenia 1, 4 i 10.
- **Rozszerzenie na kolejne zajęcia:** zamień miniaturowe macierze na `torch.nn.TransformerEncoderLayer` albo model już zapisany lokalnie. Nie planuj pobierania wag podczas zajęć.

### Dokumentacja bibliotek

- [scikit-learn: MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)
- [scikit-learn: GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- [pandas: read_csv i chunksize](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [Dask DataFrame i compute](https://docs.dask.org/en/latest/dataframe.html)
- [DuckDB: bezpośredni odczyt CSV](https://duckdb.org/docs/stable/data/csv/overview)
- [Vaswani et al., Attention Is All You Need](https://arxiv.org/abs/1706.03762)
